# 04 — Reject Inference: Correcting Selection Bias

When a lender only observes outcomes for approved applicants, the training data suffers from **selection bias**.  
Rejected applicants — who may have defaulted at higher rates — are invisible to the model.

**Approach: Parcelling (Simple Augmentation)**  
1. Train a model on accepted loans  
2. Score rejected applicants with it  
3. Assign pseudo-labels based on score (high score → likely default)  
4. Retrain on the combined dataset with soft labels

This is a simplified approach. Production methods include EM-based techniques and Heckman correction.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve

plt.style.use('seaborn-v0_8-whitegrid')

SILVER_DIR = Path("../data/silver")
GOLD_DIR = Path("../data/gold")
MODELS_DIR = Path("../data/models")

# Load accepted (Gold) data
with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)
feature_cols = meta["feature_columns"]

train = pd.read_parquet(GOLD_DIR / "features_train.parquet")
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")

# Load rejected (Silver) data
rejected = pd.read_parquet(SILVER_DIR / "rejected_clean.parquet")

# Load champion model (trained on accepted only)
with open(MODELS_DIR / "champion" / "model.pkl", "rb") as f:
    accepted_model = pickle.load(f)

print(f"Accepted train: {len(train):,} rows")
print(f"Rejected: {len(rejected):,} rows")
print(f"\nAccepted features: {feature_cols[:5]}...")
print(f"Rejected features: {rejected.columns.tolist()}")

## 1. Align Rejected Data to Accepted Feature Space

Rejected applicants have fewer columns. We use the overlap (FICO, DTI, loan amount, emp_length) and fill the rest with population medians from the accepted training set.

In [ ]:
# Sample rejected to avoid memory issues (take 500K)
np.random.seed(42)
rejected_sample = rejected.sample(n=min(500_000, len(rejected)), random_state=42).copy()

# Build a feature-aligned dataframe for rejected applicants
# Overlapping features: fico_score, dti, loan_amnt, emp_length, emp_length_missing
rejected_aligned = pd.DataFrame(index=rejected_sample.index)

# Direct mappings
rejected_aligned['fico_score'] = rejected_sample['fico_score']
rejected_aligned['dti'] = rejected_sample['dti']
rejected_aligned['loan_amnt'] = rejected_sample['loan_amnt']
rejected_aligned['emp_length'] = rejected_sample['emp_length']
rejected_aligned['emp_length_missing'] = rejected_sample['emp_length_missing']

# Fill remaining features with training set medians (conservative imputation)
train_medians = train[feature_cols].median()
for col in feature_cols:
    if col not in rejected_aligned.columns:
        rejected_aligned[col] = train_medians[col]

# Ensure column order matches
rejected_aligned = rejected_aligned[feature_cols]

print(f"Rejected aligned: {rejected_aligned.shape}")
print(f"Null check: {rejected_aligned.isnull().sum().sum()} nulls")

# Fill any remaining NaN
rejected_aligned = rejected_aligned.fillna(train_medians)

## 2. Score Rejected Applicants & Assign Pseudo-Labels

In [ ]:
# Score rejected applicants using the accepted-only model
rejected_scores = accepted_model.predict_proba(rejected_aligned)[:, 1]

# Parcelling: assign hard pseudo-labels based on score
# Use a higher threshold than the training default rate (rejected pop is riskier)
training_default_rate = train['default'].mean()
print(f"Training default rate (accepted): {training_default_rate:.3f}")

# Rejected applicants were turned down — assume they would default at ~2x the accepted rate
reject_default_rate = min(training_default_rate * 2.0, 0.60)
threshold = np.percentile(rejected_scores, 100 * (1 - reject_default_rate))
print(f"Assumed reject default rate: {reject_default_rate:.3f}")
print(f"Score threshold for pseudo-default: {threshold:.4f}")

rejected_aligned['default'] = (rejected_scores >= threshold).astype(int)

print(f"\nPseudo-label distribution:")
print(f"  Non-default: {(rejected_aligned['default'] == 0).sum():,}")
print(f"  Default:     {(rejected_aligned['default'] == 1).sum():,}")
print(f"  Rate:        {rejected_aligned['default'].mean():.3f}")

# Compare score distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(accepted_model.predict_proba(train[feature_cols])[:, 1], bins=50, alpha=0.5, 
        density=True, label='Accepted (known outcomes)', color='#2ecc71')
ax.hist(rejected_scores, bins=50, alpha=0.5, 
        density=True, label='Rejected (pseudo-labels)', color='#e74c3c')
ax.axvline(threshold, color='black', linestyle='--', label=f'Pseudo-label threshold ({threshold:.3f})')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution: Accepted vs Rejected Applicants')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Retrain on Combined Dataset & Compare

In [ ]:
# Combine accepted + rejected (with pseudo-labels)
# Weight rejected samples lower (0.3) since labels are uncertain
X_combined = pd.concat([train[feature_cols], rejected_aligned[feature_cols]], ignore_index=True)
y_combined = pd.concat([train['default'], rejected_aligned['default']], ignore_index=True)

sample_weights = np.concatenate([
    np.ones(len(train)),                      # accepted: weight 1.0
    np.full(len(rejected_aligned), 0.3),      # rejected: weight 0.3 (uncertain labels)
])

print(f"Combined dataset: {len(X_combined):,} rows")
print(f"  Accepted: {len(train):,} (weight=1.0)")
print(f"  Rejected: {len(rejected_aligned):,} (weight=0.3)")

# Train augmented model
augmented_model = HistGradientBoostingClassifier(
    max_iter=1000,
    max_depth=6,
    learning_rate=0.05,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=50,
    scoring="roc_auc",
    random_state=42,
    verbose=0,
)

augmented_model.fit(X_combined, y_combined, sample_weight=sample_weights)
print(f"Augmented model iterations: {augmented_model.n_iter_}")